## Final Workflow: ML based integration pipeline (Token Blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd

amazon_books = pd.read_parquet(DATA_DIR / "amazon_df.parquet")
metabooks = pd.read_parquet(DATA_DIR / "metabooks_df.parquet")
goodreads = pd.read_parquet(DATA_DIR / "goodreads_df.parquet")

### Sample Datasets

In [3]:
amazon_books.shape, metabooks.shape, goodreads.shape

((271360, 12), (535159, 12), (52478, 12))

In [4]:
isbn_m2a = set(metabooks['isbn_clean']) & set(amazon_books['isbn_clean'])

In [5]:
isbn_m2g = set(metabooks['isbn_clean']) & set(goodreads['isbn_clean'])

In [ ]:
metabooks_sample = metabooks[metabooks['isbn_clean'].isin(isbn_m2a)]
amazon_sample = amazon_books[amazon_books['isbn_clean'].isin(isbn_m2a)]
goodreads_sample = goodreads[goodreads['isbn_clean'].isin(isbn_m2g)]

In [8]:
az_sample_2 = amazon_books[~amazon_books['isbn_clean'].isin(isbn_m2g)].sample(30_000 - len(amazon_sample), random_state=42)
amazon_sample = pd.concat([amazon_sample, az_sample_2]).reset_index(drop=True)

In [9]:
mb_sample_2 = metabooks[~metabooks['isbn_clean'].isin(isbn_m2g)].sample(30_000 - len(metabooks_sample), random_state=42)
metabooks_sample = pd.concat([metabooks_sample, mb_sample_2]).reset_index(drop=True)

In [10]:
gr_sample_2 = goodreads[~goodreads['isbn_clean'].isin(isbn_m2g)].sample(30_000 - len(goodreads_sample), random_state=42)
goodreads_sample = pd.concat([goodreads_sample, gr_sample_2]).reset_index(drop=True)

In [11]:
metabooks_sample.shape, amazon_sample.shape, goodreads_sample.shape

((30000, 12), (30000, 12), (30000, 12))

### Entity Matching

In [12]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [13]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author_name',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author_name',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author_name',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    
    StringComparator(column='isbn_clean',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='levenshtein', preprocess=str.lower),

    NumericComparator(column='publish_year',max_difference=1),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower, 
                     list_strategy='concatenate'),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower,
                     list_strategy='best_match'),

    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2)
]

/Users/abd/Developer/team_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Token Blocker

In [ ]:
from PyDI.entitymatching import TokenBlocker


blocker_m2a = TokenBlocker(
    metabooks_sample, amazon_sample,
    column='title',
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=3,
    ngram_type='character'
)

blocker_m2g = TokenBlocker(
    metabooks_sample, goodreads_sample,
    column='title',
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=3,
    ngram_type='character'
)

token_candidates_m2a = blocker_m2a.materialize()
token_candidates_m2g = blocker_m2g.materialize()

### Evaluate Blocking

In [ ]:
from PyDI.entitymatching import EntityMatchingEvaluator


# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=token_candidates_m2a,
    blocker=blocker_m2a,
    test_pairs=test_m2a,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2a)

{'pair_completeness': 0.9024390243902439,
 'pair_quality': 5.496907024126073e-06,
 'reduction_ratio': 0.9999536494738233,
 'total_candidates': 6731058,
 'total_possible_pairs': 145220746240,
 'true_positives_found': 37,
 'total_true_pairs': 41,
 'evaluation_timestamp': '2025-11-09T17:45:35.610252',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [ ]:
# evaluate blocking: metabooks -> goodreads
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=token_candidates_m2g,
    blocker=blocker_m2g,
    test_pairs=test_m2g,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2g)

{'pair_completeness': 0.8863636363636364,
 'pair_quality': 2.1702451597968205e-05,
 'reduction_ratio': 0.9999360124175761,
 'total_candidates': 1797032,
 'total_possible_pairs': 28084074002,
 'true_positives_found': 39,
 'total_true_pairs': 44,
 'evaluation_timestamp': '2025-11-09T17:47:35.918557',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

### ML-based Matcher

In [43]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor = FeatureExtractor(comparators)

train_features_m2a = feature_extractor.create_features(
    metabooks, amazon_books, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks, goodreads, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [ ]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a = ml_matcher.match(
    metabooks_sample, amazon_sample,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g = ml_matcher.match(
    metabooks_sample, goodreads_sample,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [ ]:
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a_tbsample.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g_tbsample.csv", index=False)

### Evaluate Matching

In [48]:
debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

# evaluate matching metabooks -> amazon
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2a,
    test_pairs=test_m2a,
    out_dir=debug_output_dir
)

display(eval_results_m2a)

{'precision': 1.0,
 'recall': 0.8536585365853658,
 'f1': 0.9210526315789475,
 'accuracy': 0.97,
 'true_positives': 35,
 'false_positives': 0,
 'false_negatives': 6,
 'true_negatives': 159,
 'threshold_used': 0.0,
 'total_correspondences': 25898,
 'filtered_correspondences': 25898,
 'evaluation_timestamp': '2025-11-09T19:17:32.597347',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [49]:
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 0.918918918918919,
 'recall': 0.7727272727272727,
 'f1': 0.8395061728395061,
 'accuracy': 0.935,
 'true_positives': 34,
 'false_positives': 3,
 'false_negatives': 10,
 'true_negatives': 153,
 'threshold_used': 0.0,
 'total_correspondences': 5916,
 'filtered_correspondences': 5916,
 'evaluation_timestamp': '2025-11-09T19:17:44.346399',
 'output_files': ['/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project_data_integration/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

### Cluster Analysis

In [ ]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)

In [ ]:
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

### Data Fusion

In [ ]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

correspondences_m2a = pd.read_csv(CORR_DIR / "workflow_corr_m2a_tbsample.csv")
correspondences_m2g = pd.read_csv(CORR_DIR / "workflow_corr_m2g_tbsample.csv")

all_correspondences = pd.concat([correspondences_m2a, correspondences_m2g], ignore_index=True)

len(all_correspondences)

31814

In [ ]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author_name', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [52]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_ml_snblocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_ml_snblocker):,}')
display(fused_ml_snblocker.head())

Fused rows: 30,185


,_id,_fusion_group_id,_fusion_sources,genres,numratings,page_count,publisher,price,edition,language,rating,isbn_clean,publish_year,author,id,bookformat,title,_fusion_confidence,_fusion_metadata
0,amazon_50982,group_0,"[metabooks, amazon_books]","[['Politics', 'Social Sciences', 'Sociology']]",34,384.0,Owl Books (NY),18.400000,None,English,4.4,0805067744,2000.0,susan strasser,amazon_50982,None,never done a history of american housework,0.692308,"{'genres_rule': 'union', 'genres_sources': ['m..."
1,amazon_8369,group_1,"[metabooks, amazon_books]","[['Mystery, Thriller', 'Suspense', 'Mystery']]",44,288.0,Scribner,24.020000,None,English,3.9,074321269X,2001.0,denise hamilton,amazon_8369,None,the jasmine trade a novel of suspense introduc...,0.692308,"{'genres_rule': 'union', 'genres_sources': ['m..."
2,metabooks_251675,group_2,"[metabooks, amazon_books]","[['Literature', 'Fiction', 'Action', 'Adventur...",773,368.0,William Morrow,13.170000,None,English,4.5,0060094117,2004.0,dale brown,metabooks_251675,None,plan of attack a novel patrick mclanahan,0.699038,"{'genres_rule': 'union', 'genres_sources': ['m..."
3,goodreads_22442,group_3,"[goodreads, metabooks]","[['Fiction', 'Queer', 'LGBT', 'Transgender', '...",3370,262.0,Topside Press,12.890000,nan,English,4.3,0983242291,2013.0,imogen binnie goodreads author,goodreads_22442,Paperback,nevada,0.791026,"{'genres_rule': 'union', 'genres_sources': ['g..."
4,metabooks_434699,group_4,"[metabooks, amazon_books]","[['Politics', 'Social Sciences']]",86,600.0,W H Freeman &amp; Co,18.540001,None,English,4.4,0312101597,1996.0,chava frankfortnachmias,metabooks_434699,None,research methods in the social sciences 5th ed...,0.701357,"{'genres_rule': 'union', 'genres_sources': ['m..."


In [53]:
fused_ml_snblocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30185 entries, 0 to 30184
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 30185 non-null  object 
 1   _fusion_group_id    30185 non-null  object 
 2   _fusion_sources     30185 non-null  object 
 3   genres              28964 non-null  object 
 4   numratings          30185 non-null  int64  
 5   page_count          27473 non-null  float64
 6   publisher           30185 non-null  object 
 7   price               27975 non-null  float64
 8   edition             5849 non-null   object 
 9   language            30069 non-null  object 
 10  rating              30185 non-null  float64
 11  isbn_clean          30185 non-null  object 
 12  publish_year        30171 non-null  float64
 13  author              30185 non-null  object 
 14  id                  30185 non-null  object 
 15  bookformat          5849 non-null   object 
 16  titl

In [54]:
fused_ml_snblocker.to_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")

### Fusion Evaluation

In [55]:
fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")

In [56]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd

def _is_missing_value(v):
    if v is None:
        return True
    if isinstance(v, float) and np.isnan(v):
        return True
    if isinstance(v, (list, tuple, set, np.ndarray)):
        return len(v) == 0
    if isinstance(v, str):
        s = v.strip().lower()
        return s in ("", "nan", "none")
    return False

def array_set_equality_match(fused_value, gold_value) -> bool:
    """Case-insensitive set equality for strings, arrays, or stringified lists."""
    if _is_missing_value(fused_value) and _is_missing_value(gold_value):
        return True
    if _is_missing_value(fused_value) or _is_missing_value(gold_value):
        return False

    def to_clean_set(value):
        # unwrap 0-d or 1-element numpy arrays
        if isinstance(value, np.ndarray):
            # flatten and collapse 1-element arrays
            flat = value.flatten().tolist()
            if len(flat) == 1:
                value = flat[0]
            else:
                value = flat

        # parse stringified lists like "['Fiction','Drama']"
        if isinstance(value, str):
            s = value.strip()
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    value = parsed
                else:
                    # simple delimited string
                    value = re.split(r"[|,;/]", s)
            except Exception:
                # not parsable, split manually
                value = re.split(r"[|,;/]", s)

        # flatten iterable containers
        if isinstance(value, np.ndarray):
            value = value.tolist()
        if isinstance(value, (list, tuple, set)):
            items = []
            for v in value:
                # recursively parse if element looks like "['a','b']"
                if isinstance(v, str) and v.strip().startswith("["):
                    try:
                        inner = ast.literal_eval(v)
                        if isinstance(inner, (list, tuple, set)):
                            items.extend(inner)
                            continue
                    except Exception:
                        pass
                items.append(v)
            return {str(x).strip().lower() for x in items if not _is_missing_value(x)}

        # scalar fallback
        return {str(value).strip().lower()}

    fused_set = to_clean_set(fused_value)
    gold_set  = to_clean_set(gold_value)

    if not fused_set and not gold_set:
        return True
    return fused_set == gold_set



strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("rating", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("genres", array_set_equality_match)





In [57]:
dupe_rows = fused_dataset[fused_dataset['isbn_clean'].duplicated(keep=False)]
dupe_rows.sort_values('isbn_clean')


,_id,_fusion_group_id,_fusion_sources,genres,numratings,page_count,publisher,price,edition,language,rating,isbn_clean,publish_year,author,id,bookformat,title,_fusion_confidence,_fusion_metadata
580,metabooks_323815,group_580,"[goodreads, metabooks]","[['Picture Books', 'Childrens', 'Animals', 'Fi...",1,24,Golden Press,7.98,nan,English,5.0,0307021459,1990,annie north bedford samuel armstrong adaptor,metabooks_323815,Paperback,donald ducks toy sailboat,0.791084,"{'_id_inputs': [{'dataset': 'metabooks', 'reco..."
10871,amazon_60391,group_10871,"[metabooks, amazon_books]","[[""Children's Books""]]",26,<NA>,Western Pub. Co,7.79,None,English,4.5,0307021459,1993,annie north bedford,amazon_60391,None,walt disneys donald ducks toy sailboat a littl...,0.615385,"{'_id_inputs': [{'dataset': 'amazon_books', 'r..."
19555,goodreads_32021,group_19555,"[goodreads, metabooks]","[['Fiction', 'Historical Fiction', 'Military F...",8142,352,G.P. Putnam's Sons,8.99,nan,English,4.7,0515087491,1986,w e b griffin,goodreads_32021,Paperback,semper fi the corps book 1,0.800296,"{'_id_inputs': [{'dataset': 'goodreads', 'reco..."
20543,metabooks_328776,group_20543,"[metabooks, amazon_books]",None,11,<NA>,Jove Books,6.57,None,English,4.6,0515087491,1988,w e b griffin,metabooks_328776,None,the corps semper fibk 1 corps paperback,0.558185,"{'_id_inputs': [{'dataset': 'metabooks', 'reco..."


In [58]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.636
  macro_accuracy: 0.638
  num_evaluated_records: 38828
  num_evaluated_attributes: 9
  total_evaluations: 342147
  total_correct: 217741
  genres_accuracy: 0.640
  genres_count: 37448
  numratings_accuracy: 0.690
  numratings_count: 38828
  publisher_accuracy: 0.341
  publisher_count: 38828
  price_accuracy: 0.756
  price_count: 36440
  page_count_accuracy: 0.678
  page_count_count: 35411
  rating_accuracy: 0.630
  rating_count: 38828
  publish_year_accuracy: 0.745
  publish_year_count: 38708
  author_accuracy: 0.760
  author_count: 38828
  title_accuracy: 0.499
  title_count: 38828

Overall Accuracy: 63.6%
